In [1]:
import os
import time
import gc
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models

print("[OK] All imports successful")
print(f"PyTorch version: {torch.__version__}")


[OK] All imports successful
PyTorch version: 2.10.0+cu128


In [2]:
import os

KAGGLE_INPUT = "/kaggle/input"

competition_path = os.path.join(KAGGLE_INPUT, "competitions", "plant-leaves-super-resolution-challenge")

if os.path.exists(competition_path):
    DATASET_PATH = competition_path
else:
    
    dataset_folders = os.listdir(KAGGLE_INPUT)
    DATASET_PATH = None
    for folder in dataset_folders:
        candidate = os.path.join(KAGGLE_INPUT, folder)
        if os.path.isdir(os.path.join(candidate, "train_Low_Resolution")):
            DATASET_PATH = candidate
            break

assert DATASET_PATH is not None, f"Dataset not found. Folders: {os.listdir(KAGGLE_INPUT)}"

print(f"[INFO] Dataset found at: {DATASET_PATH}")

TRAIN_LR  = os.path.join(DATASET_PATH, "train_Low_Resolution")
TRAIN_HR  = os.path.join(DATASET_PATH, "train_High_Resolution")
TEST_LR   = os.path.join(DATASET_PATH, "test_Low_Resolution")
VGG_PATH  = os.path.join(DATASET_PATH, "vgg19_weights.pth")

# Verify
for p in [TRAIN_LR, TRAIN_HR, TEST_LR, VGG_PATH]:
    print(f"[CHECK] {p} →", "OK" if os.path.exists(p) else "MISSING")


[INFO] Dataset found at: /kaggle/input/competitions/plant-leaves-super-resolution-challenge
[CHECK] /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution → OK
[CHECK] /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_High_Resolution → OK
[CHECK] /kaggle/input/competitions/plant-leaves-super-resolution-challenge/test_Low_Resolution → OK
[CHECK] /kaggle/input/competitions/plant-leaves-super-resolution-challenge/vgg19_weights.pth → OK


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[GPU] {torch.cuda.get_device_name(0)}")
    print(f"[VRAM] {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[CPU] No GPU found — running on CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

g = torch.Generator()
g.manual_seed(SEED)

print(f"[OK] Device: {device}, Seed: {SEED}")


[GPU] Tesla T4
[VRAM] 15.6 GB
[OK] Device: cuda, Seed: 42


In [4]:
WARMUP_EPOCHS    = 40
GAN_EPOCHS       = 110
TOTAL_EPOCHS     = WARMUP_EPOCHS + GAN_EPOCHS

BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 4
LR_G             = 1e-4
LR_D             = 1e-4
BETAS            = (0.9, 0.999)

LAMBDA_PIXEL      = 1.0
LAMBDA_PERCEPTUAL = 0.006
LAMBDA_ADV        = 0.001

PATIENCE          = 30
RUNTIME_LIMIT_HRS = 8.0

LR_SIZE  = 32
HR_SIZE  = 128
CHANNELS = 3
NUM_RRDB = 16
D_UPDATE_FREQ = 1
LR_MIN = 1e-6 

CHECKPOINT_PATH = "/kaggle/working/best_generator.pth"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

USE_AMP = torch.cuda.is_available()

print(f"[CONFIG] Warmup={WARMUP_EPOCHS}, GAN={GAN_EPOCHS}, Total={TOTAL_EPOCHS}")
print(f"[CONFIG] Batch={BATCH_SIZE}, GradAccum={GRAD_ACCUM_STEPS}, EffBatch={BATCH_SIZE*GRAD_ACCUM_STEPS}")
print(f"[CONFIG] LR_G={LR_G}, LR_D={LR_D}, AMP={USE_AMP}")
print(f"[CONFIG] Patience={PATIENCE}, RuntimeLimit={RUNTIME_LIMIT_HRS}h")
print(f"[CONFIG] RRDB Blocks={NUM_RRDB}")

[CONFIG] Warmup=40, GAN=110, Total=150
[CONFIG] Batch=8, GradAccum=4, EffBatch=32
[CONFIG] LR_G=0.0001, LR_D=0.0001, AMP=True
[CONFIG] Patience=30, RuntimeLimit=8.0h
[CONFIG] RRDB Blocks=16


In [5]:
class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, lr_size=LR_SIZE, hr_size=HR_SIZE, augment=False):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.augment = augment
        self.lr_files = sorted([f for f in os.listdir(lr_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
        assert len(self.lr_files) > 0, f"No images found in {lr_dir}"
        print(f"[Dataset] Found {len(self.lr_files)} LR images in {lr_dir}")

    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        lr_name = self.lr_files[idx]
        lr_path = os.path.join(self.lr_dir, lr_name)
        hr_name = lr_name.replace("_LR", "").replace("_lr", "")
        hr_path = os.path.join(self.hr_dir, hr_name)

        try:
            lr_img = Image.open(lr_path).convert("RGB")
            hr_img = Image.open(hr_path).convert("RGB")
        except Exception as e:
            print(f"[WARN] Skipping sample {lr_name}: {e}")
            return None

        lr_img = lr_img.resize((self.lr_size, self.lr_size), Image.BILINEAR)
        hr_img = hr_img.resize((self.hr_size, self.hr_size), Image.BILINEAR)

        # Augmentation — same random transform applied to both LR and HR
        if self.augment:
            if random.random() > 0.5:
                lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
                hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                lr_img = lr_img.transpose(Image.FLIP_TOP_BOTTOM)
                hr_img = hr_img.transpose(Image.FLIP_TOP_BOTTOM)
            angle = random.choice([0, 90, 180, 270])
            if angle != 0:
                lr_img = lr_img.rotate(angle)
                hr_img = hr_img.rotate(angle)

        lr_tensor = torch.from_numpy(np.array(lr_img, dtype=np.float32) / 127.5 - 1.0).permute(2, 0, 1)
        hr_tensor = torch.from_numpy(np.array(hr_img, dtype=np.float32) / 127.5 - 1.0).permute(2, 0, 1)

        return lr_tensor, hr_tensor


def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None, None
    return torch.utils.data.dataloader.default_collate(batch)


full_dataset = SRDataset(TRAIN_LR, TRAIN_HR, augment=False)  # augment set below per split
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [train_size, val_size], generator=g
)

# Train split gets augmentation, val split does not
train_dataset = SRDataset(TRAIN_LR, TRAIN_HR, augment=True)
train_dataset.lr_files = [full_dataset.lr_files[i] for i in train_indices.indices]

val_dataset = SRDataset(TRAIN_LR, TRAIN_HR, augment=False)
val_dataset.lr_files = [full_dataset.lr_files[i] for i in val_indices.indices]

print(f"[OK] Train: {len(train_dataset)} samples (augmented), Val: {len(val_dataset)} samples")

[Dataset] Found 1642 LR images in /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution
[Dataset] Found 1642 LR images in /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution
[Dataset] Found 1642 LR images in /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution
[OK] Train: 1477 samples (augmented), Val: 165 samples


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    worker_init_fn=seed_worker,
    generator=g,
    collate_fn=safe_collate
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    worker_init_fn=seed_worker,
    generator=g,
    collate_fn=safe_collate
)

In [7]:
class DenseBlock(nn.Module):
    def __init__(self, in_channels=64, growth_rate=32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, growth_rate, 3, padding=1)
        self.conv2 = nn.Conv2d(in_channels + growth_rate * 1, growth_rate, 3, padding=1)
        self.conv3 = nn.Conv2d(in_channels + growth_rate * 2, growth_rate, 3, padding=1)
        self.conv4 = nn.Conv2d(in_channels + growth_rate * 3, growth_rate, 3, padding=1)
        self.conv5 = nn.Conv2d(in_channels + growth_rate * 4, in_channels, 3, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], 1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], 1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], 1))
        return x5 * 0.2 + x


class RRDB(nn.Module):
    def __init__(self, in_channels=64, growth_rate=32):
        super().__init__()
        self.db1 = DenseBlock(in_channels, growth_rate)
        self.db2 = DenseBlock(in_channels, growth_rate)
        self.db3 = DenseBlock(in_channels, growth_rate)

    def forward(self, x):
        out = self.db1(x)
        out = self.db2(out)
        out = self.db3(out)
        return out * 0.2 + x


print("[OK] DenseBlock and RRDB defined")

[OK] DenseBlock and RRDB defined


In [8]:
class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_features=64, num_rrdb=16):
        super().__init__()
        self.conv_first = nn.Conv2d(in_channels, num_features, 3, padding=1)
        self.trunk = nn.Sequential(*[RRDB(num_features, 32) for _ in range(num_rrdb)])
        self.conv_last_trunk = nn.Conv2d(num_features, num_features, 3, padding=1)

        self.up1_conv = nn.Conv2d(num_features, num_features * 4, 3, padding=1)
        self.up1_ps = nn.PixelShuffle(2)
        self.up2_conv = nn.Conv2d(num_features, num_features * 4, 3, padding=1)
        self.up2_ps = nn.PixelShuffle(2)

        self.hr_conv = nn.Conv2d(num_features, num_features, 3, padding=1)
        self.conv_last = nn.Conv2d(num_features, out_channels, 3, padding=1)

        self.skip_hr_conv = nn.Conv2d(num_features, num_features, 3, padding=1)
        self.skip_conv_last = nn.Conv2d(num_features, out_channels, 3, padding=1)

        self.lrelu = nn.LeakyReLU(0.2)
        self.tanh = nn.Tanh()

    def forward(self, x):
        feat_first = self.conv_first(x)
        trunk_out = self.conv_last_trunk(self.trunk(feat_first))
        feat = feat_first + trunk_out

        feat = F.leaky_relu(self.up1_ps(self.up1_conv(feat)), negative_slope=0.2, inplace=False)
        feat = F.leaky_relu(self.up2_ps(self.up2_conv(feat)), negative_slope=0.2, inplace=False)
        feat = F.leaky_relu(self.hr_conv(feat), negative_slope=0.2, inplace=False)
        out = self.tanh(self.conv_last(feat))

        skip = F.interpolate(feat_first, scale_factor=4, mode="bilinear", align_corners=False)
        skip = F.leaky_relu(self.skip_hr_conv(skip), negative_slope=0.2, inplace=False)
        skip = self.tanh(self.skip_conv_last(skip))

        return torch.clamp(out + skip, -1.0, 1.0)


def init_weights(module):
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="leaky_relu")
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.constant_(module.weight, 1)
        nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, 0, 0.01)
        nn.init.constant_(module.bias, 0)


generator = Generator(num_rrdb=NUM_RRDB).to(device)
generator.apply(init_weights)

total_params_g = sum(p.numel() for p in generator.parameters())
print(f"[OK] Generator created: {total_params_g / 1e6:.2f}M parameters")

[OK] Generator created: 11.92M parameters


In [9]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3, num_filters=64):
        super().__init__()

        def block(in_c, out_c, stride, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, stride=stride, padding=1, bias=not norm)]
            if norm:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=False))
            return layers

        self.model = nn.Sequential(
            *block(in_channels,      num_filters,     stride=2, norm=False),  # 64x64
            *block(num_filters,      num_filters * 2, stride=2),              # 32x32
            *block(num_filters * 2,  num_filters * 4, stride=2),              # 16x16
            *block(num_filters * 4,  num_filters * 8, stride=1),              # 16x16
            nn.Conv2d(num_filters * 8, 1, 4, stride=1, padding=1)             # patch map
        )

    def forward(self, x):
        return self.model(x)


discriminator = Discriminator().to(device)
discriminator.apply(init_weights)

total_params_d = sum(p.numel() for p in discriminator.parameters())
print(f"[OK] PatchGAN Discriminator created: {total_params_d / 1e6:.2f}M parameters")

[OK] PatchGAN Discriminator created: 2.77M parameters


In [10]:
class VGGPerceptualLoss(nn.Module):
    def __init__(self, vgg_path):
        super().__init__()
        vgg = models.vgg19(pretrained=False)
        vgg.load_state_dict(torch.load(vgg_path, map_location='cpu'))


        self.feature_extractor = nn.Sequential(*list(vgg.features)[:27])
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        self.feature_extractor.eval()

        self.register_buffer('vgg_mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('vgg_std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, generated, target):
        gen_vgg = (generated + 1.0) / 2.0
        tgt_vgg = (target + 1.0) / 2.0
        gen_vgg = (gen_vgg - self.vgg_mean) / self.vgg_std
        tgt_vgg = (tgt_vgg - self.vgg_mean) / self.vgg_std
        gen_feat = self.feature_extractor(gen_vgg)
        tgt_feat = self.feature_extractor(tgt_vgg)
        return F.mse_loss(gen_feat, tgt_feat)


perceptual_loss_fn = VGGPerceptualLoss(VGG_PATH).to(device)
print("[OK] VGGPerceptualLoss loaded — using relu4_3 (layer 27)")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


[OK] VGGPerceptualLoss loaded — using relu4_3 (layer 27)


In [11]:
criterion_pixel = nn.L1Loss().to(device)
criterion_adv   = nn.BCEWithLogitsLoss().to(device)

optimizer_G = optim.Adam(generator.parameters(),     lr=LR_G, betas=BETAS)
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR_D, betas=BETAS)

scheduler_G = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_G, T_max=WARMUP_EPOCHS, eta_min=LR_MIN
)

scaler_G = torch.amp.GradScaler('cuda', enabled=USE_AMP)
scaler_D = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print("[OK] Losses, optimizers, scheduler, and AMP scalers initialized")

[OK] Losses, optimizers, scheduler, and AMP scalers initialized


In [12]:
print("=" * 60)
print("BATCH DIAGNOSTICS: Checking for NaN/Inf before training")
print("=" * 60)

generator.eval()
discriminator.eval()
with torch.no_grad():
    lr_batch, hr_batch = next(iter(train_loader))
    if lr_batch is None:
        raise RuntimeError("No valid samples in first diagnostic batch.")

    lr_batch = lr_batch.to(device)
    hr_batch = hr_batch.to(device)

    print(f"\n[LR BATCH] Shape: {lr_batch.shape}")
    print(f"  Range: [{lr_batch.min():.4f}, {lr_batch.max():.4f}]")
    print(f"  Mean: {lr_batch.mean():.4f}, Std: {lr_batch.std():.4f}")
    print(f"  Has NaN: {torch.isnan(lr_batch).any()}, Has Inf: {torch.isinf(lr_batch).any()}")

    print(f"\n[HR BATCH] Shape: {hr_batch.shape}")
    print(f"  Range: [{hr_batch.min():.4f}, {hr_batch.max():.4f}]")
    print(f"  Mean: {hr_batch.mean():.4f}, Std: {hr_batch.std():.4f}")
    print(f"  Has NaN: {torch.isnan(hr_batch).any()}, Has Inf: {torch.isinf(hr_batch).any()}")

    with torch.amp.autocast("cuda", enabled=USE_AMP):
        sr_batch = generator(lr_batch)
    sr_batch = sr_batch.float()
    
    print(f"\n[SR BATCH] Shape: {sr_batch.shape}")
    print(f"  Range: [{sr_batch.min():.4f}, {sr_batch.max():.4f}]")
    print(f"  Mean: {sr_batch.mean():.4f}, Std: {sr_batch.std():.4f}")
    print(f"  Has NaN: {torch.isnan(sr_batch).any()}, Has Inf: {torch.isinf(sr_batch).any()}")

    loss_pixel = criterion_pixel(sr_batch, hr_batch)
    loss_perc = perceptual_loss_fn(sr_batch, hr_batch)
    pred_real = discriminator(hr_batch)
    pred_fake = discriminator(sr_batch)
    loss_adv = criterion_adv(pred_fake, torch.ones_like(pred_fake, device=device))

    print(f"\n[LOSS VALUES]")
    print(f"  L1/Pixel: {loss_pixel.item():.6f}, NaN: {torch.isnan(loss_pixel)}, Inf: {torch.isinf(loss_pixel)}")
    print(f"  Perceptual: {loss_perc.item():.6f}, NaN: {torch.isnan(loss_perc)}, Inf: {torch.isinf(loss_perc)}")
    print(f"  Adversarial: {loss_adv.item():.6f}, NaN: {torch.isnan(loss_adv)}, Inf: {torch.isinf(loss_adv)}")
    print(f"  Disc Real: {pred_real.mean():.6f}, range: [{pred_real.min():.4f}, {pred_real.max():.4f}]")
    print(f"  Disc Fake: {pred_fake.mean():.6f}, range: [{pred_fake.min():.4f}, {pred_fake.max():.4f}]")

generator.train()
discriminator.train()
print("\n[OK] Batch diagnostics complete")

BATCH DIAGNOSTICS: Checking for NaN/Inf before training

[LR BATCH] Shape: torch.Size([8, 3, 32, 32])
  Range: [-1.0000, 0.9843]
  Mean: -0.2019, Std: 0.3513
  Has NaN: False, Has Inf: False

[HR BATCH] Shape: torch.Size([8, 3, 128, 128])
  Range: [-1.0000, 1.0000]
  Mean: -0.1976, Std: 0.3780
  Has NaN: False, Has Inf: False

[SR BATCH] Shape: torch.Size([8, 3, 128, 128])
  Range: [-1.0000, 1.0000]
  Mean: 0.2220, Std: 0.8071
  Has NaN: False, Has Inf: False

[LOSS VALUES]
  L1/Pixel: 0.872866, NaN: False, Inf: False
  Perceptual: 3.555103, NaN: False, Inf: False
  Adversarial: 1.267310, NaN: False, Inf: False
  Disc Real: -0.097572, range: [-1.9044, 1.7296]
  Disc Fake: -0.719450, range: [-5.3091, 4.2608]

[OK] Batch diagnostics complete


In [13]:
def validate_mae(gen, loader):
    gen.eval()
    total_mae = 0.0
    count = 0
    with torch.no_grad():
        for lr_batch, hr_batch in loader:
            if lr_batch is None:
                continue
            lr_batch = lr_batch.to(device, non_blocking=True)
            hr_batch = hr_batch.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                sr_batch = gen(lr_batch)
            if torch.isnan(sr_batch).any() or torch.isinf(sr_batch).any():
                gen.train()
                return float("inf")
            sr_255 = (sr_batch.float() + 1.0) * 127.5
            hr_255 = (hr_batch.float() + 1.0) * 127.5
            batch_mae = F.l1_loss(sr_255, hr_255, reduction="sum").item()
            if not np.isfinite(batch_mae):
                gen.train()
                return float("inf")
            total_mae += batch_mae
            count += hr_batch.numel()
    gen.train()
    if count == 0:
        return float("inf")
    return total_mae / count


print("=" * 60)
print("PHASE 1: Generator Warmup (pixel loss only)")
print("=" * 60)

# Remove stale checkpoint so Phase 2 doesn't accidentally load garbage
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("[INFO] Removed stale checkpoint — running Phase 1 from scratch")

train_start = time.time()
best_val_mae = float("inf")
patience_count = 0

try:
    for epoch in range(1, WARMUP_EPOCHS + 1):
        elapsed_hrs = (time.time() - train_start) / 3600
        if elapsed_hrs > RUNTIME_LIMIT_HRS:
            print(f"[TIMEOUT GUARD] {elapsed_hrs:.2f}h elapsed, stopping training early.")
            break

        warmup_steps = 5
        if epoch <= warmup_steps:
            lr_scale = epoch / warmup_steps
            for pg in optimizer_G.param_groups:
                pg["lr"] = LR_G * lr_scale

        generator.train()
        epoch_g_loss = 0.0
        num_batches = 0

        pbar = tqdm(train_loader, desc=f"Warmup Epoch {epoch}/{WARMUP_EPOCHS}")
        for batch_idx, batch in enumerate(pbar):
            lr_batch, hr_batch = batch
            if lr_batch is None:
                continue

            lr_batch = lr_batch.to(device, non_blocking=True)
            hr_batch = hr_batch.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                sr_batch = generator(lr_batch)
                if torch.isnan(sr_batch).any() or torch.isinf(sr_batch).any():
                    print(f"[WARN] NaN/Inf in generator output at batch {batch_idx}, skipping")
                    optimizer_G.zero_grad(set_to_none=True)
                    continue
                loss_pixel = criterion_pixel(sr_batch, hr_batch)
                if torch.isnan(loss_pixel) or torch.isinf(loss_pixel):
                    print(f"[WARN] Invalid pixel loss at batch {batch_idx}, skipping")
                    optimizer_G.zero_grad(set_to_none=True)
                    continue
                loss_G = (LAMBDA_PIXEL * loss_pixel) / GRAD_ACCUM_STEPS

            if not (torch.isnan(loss_G) or torch.isinf(loss_G)):
                scaler_G.scale(loss_G).backward()
            else:
                print(f"[WARN] Invalid generator loss at batch {batch_idx}, skipping backward")
                optimizer_G.zero_grad(set_to_none=True)
                continue

            if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                scaler_G.unscale_(optimizer_G)
                torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                scaler_G.step(optimizer_G)
                scaler_G.update()
                optimizer_G.zero_grad(set_to_none=True)

            epoch_g_loss += loss_pixel.item()
            num_batches += 1
            pbar.set_postfix({"pixel_loss": f"{loss_pixel.item():.4f}"})

        avg_g_loss = epoch_g_loss / max(num_batches, 1)
        val_mae = validate_mae(generator, val_loader)
        if not np.isfinite(val_mae):
            val_mae = float("inf")
        scheduler_G.step()

        elapsed_hrs = (time.time() - train_start) / 3600
        print(
            f"Epoch [{epoch}/{TOTAL_EPOCHS}] "
            f"G_loss: {avg_g_loss:.4f} | D_loss: N/A | "
            f"Val_MAE: {val_mae:.4f} | LR: {optimizer_G.param_groups[0]['lr']:.2e} | "
            f"Time: {elapsed_hrs:.2f}h"
        )

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            patience_count = 0
            torch.save(generator.state_dict(), CHECKPOINT_PATH)
            print(f"[SAVED] Best model at epoch {epoch}, MAE={val_mae:.4f}")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"[EARLY STOP] No improvement for {PATIENCE} epochs.")
                break

except Exception as e:
    print(f"[ERROR] Phase 1 crashed: {e}")
    if os.path.exists(CHECKPOINT_PATH):
        print(f"[INFO] Checkpoint exists at {CHECKPOINT_PATH}")
    else:
        torch.save(generator.state_dict(), CHECKPOINT_PATH)
        print("[EMERGENCY SAVE] Saved checkpoint after crash")

print(f"\n[Phase 1 Complete] Best Val MAE: {best_val_mae:.4f}")

PHASE 1: Generator Warmup (pixel loss only)


Warmup Epoch 1/40:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [1/150] G_loss: 0.5619 | D_loss: N/A | Val_MAE: 57.2238 | LR: 2.00e-05 | Time: 0.01h
[SAVED] Best model at epoch 1, MAE=57.2238


Warmup Epoch 2/40:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [2/150] G_loss: 0.3822 | D_loss: N/A | Val_MAE: 43.9248 | LR: 3.98e-05 | Time: 0.02h
[SAVED] Best model at epoch 2, MAE=43.9248


...

Epoch [38/150] G_loss: 0.1409 | D_loss: N/A | Val_MAE: 17.3565 | LR: 1.62e-06 | Time: 0.33h
[SAVED] Best model at epoch 38, MAE=17.3565


Warmup Epoch 39/40:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [39/150] G_loss: 0.1408 | D_loss: N/A | Val_MAE: 17.3543 | LR: 1.16e-06 | Time: 0.34h
[SAVED] Best model at epoch 39, MAE=17.3543


Warmup Epoch 40/40:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [40/150] G_loss: 0.1408 | D_loss: N/A | Val_MAE: 17.3537 | LR: 1.00e-06 | Time: 0.35h
[SAVED] Best model at epoch 40, MAE=17.3537

[Phase 1 Complete] Best Val MAE: 17.3537


In [14]:
# Memory cleanup between phases

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("[OK] Memory cleaned between Phase 1 and Phase 2")


[OK] Memory cleaned between Phase 1 and Phase 2


In [15]:
print("=" * 60)
print("PHASE 2: Full cGAN Training")
print("=" * 60)

if os.path.exists(CHECKPOINT_PATH):
    generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
generator.train()
discriminator.train()

optimizer_G = optim.Adam(generator.parameters(), lr=LR_G, betas=BETAS)
optimizer_D = optim.Adam(discriminator.parameters(), lr=1e-5, betas=BETAS)
scheduler_G = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_G, T_max=GAN_EPOCHS, eta_min=LR_MIN
)
scaler_G = torch.amp.GradScaler("cuda", enabled=USE_AMP)
scaler_D = torch.amp.GradScaler("cuda", enabled=USE_AMP)

patience_count = 0
phase2_start = time.time()

try:
    for epoch in range(WARMUP_EPOCHS + 1, TOTAL_EPOCHS + 1):
        elapsed_total_hrs = (time.time() - train_start) / 3600
        if elapsed_total_hrs > RUNTIME_LIMIT_HRS:
            print(f"[TIMEOUT GUARD] {elapsed_total_hrs:.2f}h elapsed, stopping training early.")
            break

        generator.train()
        discriminator.train()
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        num_batches = 0
        last_loss_d_real = torch.tensor(0.0, device=device)
        last_loss_d_fake = torch.tensor(0.0, device=device)

        pbar = tqdm(train_loader, desc=f"GAN Epoch {epoch}/{TOTAL_EPOCHS}")
        for batch_idx, batch in enumerate(pbar):
            lr_batch, hr_batch = batch
            if lr_batch is None:
                continue

            lr_batch = lr_batch.to(device, non_blocking=True)
            hr_batch = hr_batch.to(device, non_blocking=True)

            # ── D update: only every 2 batches ──
            if batch_idx % 2 == 0:
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    sr_detached = generator(lr_batch).detach()
                    pred_real = discriminator(hr_batch)
                    pred_fake = discriminator(sr_detached)
                    loss_d_real = criterion_adv(pred_real, torch.ones_like(pred_real))
                    loss_d_fake = criterion_adv(pred_fake, torch.zeros_like(pred_fake))
                    loss_D = (loss_d_real + loss_d_fake) / 2.0 / GRAD_ACCUM_STEPS
                    last_loss_d_real = loss_d_real.detach()
                    last_loss_d_fake = loss_d_fake.detach()

                if not (torch.isnan(loss_D) or torch.isinf(loss_D)):
                    scaler_D.scale(loss_D).backward()
                    if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                        scaler_D.unscale_(optimizer_D)
                        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
                        scaler_D.step(optimizer_D)
                        scaler_D.update()
                        optimizer_D.zero_grad(set_to_none=True)

            # ── G update: every batch ──
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                sr_batch = generator(lr_batch)
                pred_fake_for_g = discriminator(sr_batch)
                loss_pixel = criterion_pixel(sr_batch, hr_batch)
                loss_perc = perceptual_loss_fn(sr_batch, hr_batch)
                loss_adv = criterion_adv(pred_fake_for_g, torch.ones_like(pred_fake_for_g))
                loss_G = (
                    LAMBDA_PIXEL * loss_pixel
                    + LAMBDA_PERCEPTUAL * loss_perc
                    + LAMBDA_ADV * loss_adv
                ) / GRAD_ACCUM_STEPS

            if not (torch.isnan(loss_G) or torch.isinf(loss_G)):
                scaler_G.scale(loss_G).backward()
                if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                    scaler_G.unscale_(optimizer_G)
                    torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                    scaler_G.step(optimizer_G)
                    scaler_G.update()
                    optimizer_G.zero_grad(set_to_none=True)

            epoch_g_loss += (LAMBDA_PIXEL * loss_pixel + LAMBDA_PERCEPTUAL * loss_perc + 0.01 * loss_adv).item()
            epoch_d_loss += ((last_loss_d_real + last_loss_d_fake) / 2.0).item()
            num_batches += 1
            pbar.set_postfix({
                "G": f"{loss_pixel.item():.4f}",
                "D": f"{((last_loss_d_real + last_loss_d_fake) / 2.0).item():.4f}"
            })

        avg_g_loss = epoch_g_loss / max(num_batches, 1)
        avg_d_loss = epoch_d_loss / max(num_batches, 1)
        val_mae = validate_mae(generator, val_loader)
        if not np.isfinite(val_mae):
            val_mae = float("inf")
        scheduler_G.step()

        elapsed_hrs = (time.time() - train_start) / 3600
        print(
            f"Epoch [{epoch}/{TOTAL_EPOCHS}] "
            f"G_loss: {avg_g_loss:.4f} | D_loss: {avg_d_loss:.4f} | "
            f"Val_MAE: {val_mae:.4f} | LR: {optimizer_G.param_groups[0]['lr']:.2e} | "
            f"Time: {elapsed_hrs:.2f}h"
        )

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            patience_count = 0
            torch.save(generator.state_dict(), CHECKPOINT_PATH)
            print(f"[SAVED] Best model at epoch {epoch}, MAE={val_mae:.4f}")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"[EARLY STOP] No improvement for {PATIENCE} epochs.")
                break

except Exception as e:
    print(f"[ERROR] Phase 2 crashed: {e}")
    if os.path.exists(CHECKPOINT_PATH):
        print(f"[INFO] Checkpoint exists at {CHECKPOINT_PATH}")
    else:
        torch.save(generator.state_dict(), CHECKPOINT_PATH)
        print("[EMERGENCY SAVE] Saved checkpoint after crash")

print(f"\n[Phase 2 Complete] Best Val MAE: {best_val_mae:.4f}")
print(f"Total training time: {(time.time() - train_start) / 3600:.2f}h")

PHASE 2: Full cGAN Training


GAN Epoch 41/150:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [41/150] G_loss: 0.2459 | D_loss: 7.4188 | Val_MAE: 17.8482 | LR: 1.00e-04 | Time: 0.36h


GAN Epoch 42/150:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [42/150] G_loss: 0.2193 | D_loss: 7.4953 | Val_MAE: 17.6313 | LR: 9.99e-05 | Time: 0.37h


...

Epoch [148/150] G_loss: 0.1657 | D_loss: 1.4322 | Val_MAE: 16.8958 | LR: 1.08e-06 | Time: 1.60h


GAN Epoch 149/150:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [149/150] G_loss: 0.1660 | D_loss: 1.3793 | Val_MAE: 16.8960 | LR: 1.02e-06 | Time: 1.61h


GAN Epoch 150/150:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch [150/150] G_loss: 0.1659 | D_loss: 1.3877 | Val_MAE: 16.8941 | LR: 1.00e-06 | Time: 1.63h
[SAVED] Best model at epoch 150, MAE=16.8941

[Phase 2 Complete] Best Val MAE: 16.8941
Total training time: 1.63h


In [16]:
print("=" * 60)
print("PHASE 3: Fine-tune (G only, frozen D)")
print("=" * 60)

generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
generator.train()
discriminator.eval()

optimizer_G_ft = optim.Adam(generator.parameters(), lr=2e-6)
scaler_ft = torch.amp.GradScaler("cuda", enabled=USE_AMP)
best_val_mae_p3 = best_val_mae
ft_patience = 0

for epoch in range(1, 51):  # 50 epochs
    generator.train()
    for batch_idx, batch in enumerate(train_loader):
        lr_batch, hr_batch = batch
        if lr_batch is None:
            continue
        lr_batch = lr_batch.to(device)
        hr_batch = hr_batch.to(device)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            sr_batch = generator(lr_batch)
            loss = criterion_pixel(sr_batch, hr_batch) / GRAD_ACCUM_STEPS

        scaler_ft.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
            scaler_ft.unscale_(optimizer_G_ft)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), 1.0)
            scaler_ft.step(optimizer_G_ft)
            scaler_ft.update()
            optimizer_G_ft.zero_grad(set_to_none=True)

    val_mae = validate_mae(generator, val_loader)
    print(f"FT Epoch {epoch}/50 | Val_MAE: {val_mae:.4f}")

    if val_mae < best_val_mae_p3:
        best_val_mae_p3 = val_mae
        ft_patience = 0
        torch.save(generator.state_dict(), CHECKPOINT_PATH)
        print(f"[SAVED] MAE={val_mae:.4f}")
    else:
        ft_patience += 1
        if ft_patience >= 10:
            print("[EARLY STOP] No improvement for 10 epochs.")
            break

print(f"\n[Phase 3 Complete] Best MAE: {best_val_mae_p3:.4f}")

PHASE 3: Fine-tune (G only, frozen D)
FT Epoch 1/50 | Val_MAE: 16.8624
[SAVED] MAE=16.8624
FT Epoch 2/50 | Val_MAE: 16.8514
[SAVED] MAE=16.8514
FT Epoch 3/50 | Val_MAE: 16.8474
[SAVED] MAE=16.8474
FT Epoch 4/50 | Val_MAE: 16.8414
[SAVED] MAE=16.8414
FT Epoch 5/50 | Val_MAE: 16.8386
[SAVED] MAE=16.8386
FT Epoch 6/50 | Val_MAE: 16.8341
[SAVED] MAE=16.8341
FT Epoch 7/50 | Val_MAE: 16.8299
[SAVED] MAE=16.8299
FT Epoch 8/50 | Val_MAE: 16.8294
[SAVED] MAE=16.8294
FT Epoch 9/50 | Val_MAE: 16.8262
[SAVED] MAE=16.8262
FT Epoch 10/50 | Val_MAE: 16.8241
[SAVED] MAE=16.8241
FT Epoch 11/50 | Val_MAE: 16.8244
FT Epoch 12/50 | Val_MAE: 16.8198
[SAVED] MAE=16.8198
FT Epoch 13/50 | Val_MAE: 16.8223
FT Epoch 14/50 | Val_MAE: 16.8176
[SAVED] MAE=16.8176
FT Epoch 15/50 | Val_MAE: 16.8172
[SAVED] MAE=16.8172
FT Epoch 16/50 | Val_MAE: 16.8181
FT Epoch 17/50 | Val_MAE: 16.8140
[SAVED] MAE=16.8140
FT Epoch 18/50 | Val_MAE: 16.8152
FT Epoch 19/50 | Val_MAE: 16.8159
FT Epoch 20/50 | Val_MAE: 16.8124
[SAVED] MAE

In [17]:
print("=" * 60)
print("FINAL RETRAIN: Full dataset (no val split)")
print("=" * 60)

full_train_dataset = SRDataset(TRAIN_LR, TRAIN_HR, augment=True)
full_train_loader = DataLoader(
    full_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g,
    collate_fn=safe_collate
)

generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
generator.train()

optimizer_G_full = optim.Adam(generator.parameters(), lr=2e-6)
scaler_full = torch.amp.GradScaler("cuda", enabled=USE_AMP)

for epoch in range(1, 6):  # 5 epochs on full data
    generator.train()
    for batch_idx, batch in enumerate(full_train_loader):
        lr_batch, hr_batch = batch
        if lr_batch is None:
            continue
        lr_batch = lr_batch.to(device)
        hr_batch = hr_batch.to(device)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            sr_batch = generator(lr_batch)
            loss = criterion_pixel(sr_batch, hr_batch) / GRAD_ACCUM_STEPS

        scaler_full.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(full_train_loader):
            scaler_full.unscale_(optimizer_G_full)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), 1.0)
            scaler_full.step(optimizer_G_full)
            scaler_full.update()
            optimizer_G_full.zero_grad(set_to_none=True)

    print(f"[Full Retrain] Epoch {epoch}/5 done")

torch.save(generator.state_dict(), CHECKPOINT_PATH)
print("[SAVED] Full retrain checkpoint saved — ready for inference")

FINAL RETRAIN: Full dataset (no val split)
[Dataset] Found 1642 LR images in /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution
[Full Retrain] Epoch 1/5 done
[Full Retrain] Epoch 2/5 done
[Full Retrain] Epoch 3/5 done
[Full Retrain] Epoch 4/5 done
[Full Retrain] Epoch 5/5 done
[SAVED] Full retrain checkpoint saved — ready for inference


In [18]:
print("=" * 60)
print("INFERENCE: Generating super-resolved images (with TTA)")
print("=" * 60)

generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
mae_phase3 = validate_mae(generator, val_loader)

if os.path.exists("/kaggle/working/full_retrain.pth"):
    generator.load_state_dict(torch.load("/kaggle/working/full_retrain.pth", map_location=device))
    mae_full = validate_mae(generator, val_loader)
    print(f"Phase 3 MAE: {mae_phase3:.4f} | Full Retrain MAE: {mae_full:.4f}")
    if mae_full < mae_phase3:
        print("[INFO] Using full retrain checkpoint")
    else:
        print("[INFO] Using Phase 3 checkpoint")
        generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
else:
    print(f"[INFO] Using Phase 3 checkpoint | MAE: {mae_phase3:.4f}")

generator.eval()
print("[OK] Best checkpoint loaded for inference")

def infer_with_tta(gen, lr_tensor, blend=0.20):
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            p1 = gen(lr_tensor)
            p2 = torch.flip(gen(torch.flip(lr_tensor, [3])), [3])
            p3 = torch.flip(gen(torch.flip(lr_tensor, [2])), [2])
            p4 = torch.flip(gen(torch.flip(lr_tensor, [2, 3])), [2, 3])
            p5 = torch.rot90(gen(torch.rot90(lr_tensor, 1, [2, 3])), -1, [2, 3])
            p6 = torch.rot90(gen(torch.rot90(lr_tensor, 2, [2, 3])), -2, [2, 3])
            p7 = torch.rot90(gen(torch.rot90(lr_tensor, 3, [2, 3])), -3, [2, 3])
            p8 = torch.flip(torch.rot90(gen(torch.rot90(torch.flip(lr_tensor, [3]), 1, [2, 3])), -1, [2, 3]), [3])
    sr = (p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8) / 8.0
    bicubic = F.interpolate(lr_tensor.float(), scale_factor=4,
                            mode='bicubic', align_corners=False)
    bicubic = torch.clamp(bicubic, -1, 1)
    return (1 - blend) * sr + blend * bicubic

test_files = sorted(os.listdir(TEST_LR))
test_files = [f for f in test_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"[INFO] Found {len(test_files)} test images")

rows = []
for fname in tqdm(test_files, desc="Generating HR images (TTA)"):
    img_path = os.path.join(TEST_LR, fname)
    img = Image.open(img_path).convert("RGB")
    lr_tensor = torch.tensor(
        np.array(img, dtype=np.float32) / 127.5 - 1.0
    ).permute(2, 0, 1).unsqueeze(0).to(device)
    sr_tensor = infer_with_tta(generator, lr_tensor)
    sr_np = sr_tensor.squeeze(0).cpu().float().numpy()
    sr_np = (sr_np + 1.0) * 127.5
    sr_np = np.clip(sr_np, 0, 255).astype(np.uint8)
    sr_hwc = sr_np.transpose(1, 2, 0)
    flat = sr_hwc.flatten()
    assert flat.shape[0] == 49152, \
        f"[ERROR] {fname}: expected 49152 values, got {flat.shape[0]}"
    assert flat.min() >= 0 and flat.max() <= 255, \
        f"[ERROR] {fname}: pixel values out of [0,255] range"
    assert flat.dtype == np.uint8, \
        f"[ERROR] {fname}: dtype must be uint8, got {flat.dtype}"
    pixel_str = " ".join(map(str, flat.tolist()))
    rows.append({"Id": fname, "Pixels": pixel_str})

print(f"[OK] Generated {len(rows)} super-resolved images")

INFERENCE: Generating super-resolved images (with TTA)
[INFO] Using Phase 3 checkpoint | MAE: 16.7876
[OK] Best checkpoint loaded for inference
[INFO] Found 495 test images


Generating HR images (TTA):   0%|          | 0/495 [00:00<?, ?it/s]

[OK] Generated 495 super-resolved images


In [19]:
#  CSV generation with full validation

print("=" * 60)
print("CSV GENERATION & VALIDATION")
print("=" * 60)

submission_df = pd.DataFrame(rows, columns=["Id", "Pixels"])

assert len(submission_df) == 495, \
    f"[ERROR] Expected 495 rows, got {len(submission_df)}"
assert list(submission_df.columns) == ["Id", "Pixels"], \
    "[ERROR] Column names must be exactly ['Id', 'Pixels']"

sample_pixels = submission_df["Pixels"].iloc[0].split(" ")
assert len(sample_pixels) == 49152, \
    f"[ERROR] First row has {len(sample_pixels)} values, expected 49152"
assert all(v.isdigit() for v in sample_pixels[:10]), \
    "[ERROR] Pixel values must be plain integers"

print(f"[OK] submission.csv validated: {len(submission_df)} rows, 49152 values each")

submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"[SAVED] {SUBMISSION_PATH}")
print(submission_df.head(3).to_string())


CSV GENERATION & VALIDATION
[OK] submission.csv validated: 495 rows, 49152 values each
[SAVED] /kaggle/working/submission.csv
                         Id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [20]:
# Final sanity check

print("=" * 60)
print("FINAL SANITY CHECK")
print("=" * 60)

final_df = pd.read_csv(SUBMISSION_PATH)
print(f"Shape: {final_df.shape}")
print(f"Columns: {list(final_df.columns)}")
print(f"\nHead(3):")
print(final_df.head(3).to_string())

for i in range(min(3, len(final_df))):
    row_pixels = final_df["Pixels"].iloc[i].split(" ")
    print(f"  Row {i} ({final_df['Id'].iloc[i]}): {len(row_pixels)} values, "
          f"range [{min(int(v) for v in row_pixels)}-{max(int(v) for v in row_pixels)}]")

file_size_mb = os.path.getsize(SUBMISSION_PATH) / (1024 * 1024)
print(f"\n[FINAL] submission.csv size: {file_size_mb:.2f} MB")
print(f"[FINAL] Total rows: {len(final_df)}")
print(f"[FINAL] Total training time: {(time.time() - train_start) / 3600:.2f}h")
print("[DONE] Notebook execution complete!")


FINAL SANITY CHECK
Shape: (495, 2)
Columns: ['Id', 'Pixels']

Head(3):
                         Id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      